## The Cosmic Shoreline

As a fun little application of `exoatlas`, we provide an example of how to estimate the probability a planet has an atmosphere according to the 3D cosmic shoreline model presented in [Berta-Thompson, Wachiraphan, and Murray (2026)](https://ui.adsabs.harvard.edu/abs/2025arXiv250702136B/abstract). This page shows how to calculat probabilities for individual planets, for `exoatlas` populations of planets, and across a grid of parameter values; it also includes a few neat plots. 

In [2]:
from exoatlas import * 

Inference data with groups:
	> posterior

In [ ]:
from shoreline import * 

Let's read a table of planet properties grabbed from the Rocky Worlds website. 

In [ ]:
targets = QTable(ascii.read('jwst-rocky-worlds-targets.csv'))
targets['radius'] *= u.Rearth
targets['mass'] *= u.Mearth
targets['period'] *= u.day
targets['Teff'] *= u.K
targets['teq'] *= u.K 

targets

Let's translate these into the variables we need for the shoreline.

In [ ]:
R = targets['radius']
M = targets['mass']

# calculate escape velocity
relative_v = np.sqrt(M.value/R.value)
targets['log_v'] = np.log10(relative_v)

# calculate instellation
flux = targets['teq']**4*con.sigma_sb*4
earth_insolation = (1 * u.Lsun / 4 / np.pi / u.AU**2).to(u.W / u.m**2)
targets['log_f'] = np.log10(flux/earth_insolation)

# calculate stellar luminosity 
Rs = (targets['radius']/targets['Rp/R★']).to('Rsun')
L = 4*np.pi*Rs**2*con.sigma_sb*targets['Teff']**4
targets['log_L'] = np.log10(L/u.Lsun)

targets

Let load the posterior for the shoreline parameters and use it to calculate atmosphere probabilities.

In [ ]:
from exoatlas.visualizations.shorelines import * 
posterior = az.from_netcdf('cosmic-shoreline-btwm2026-posterior.nc')
shoreline = Shoreline(posterior=posterior)

predictions = targets[['planet', 'log_f', 'log_v', 'log_L']]
predictions["probability_of_atmosphere"] = (
    shoreline.calculate_probability_of_atmosphere_from_posterior_samples(
        log_v=targets["log_v"], log_L=targets["log_L"], log_f=targets["log_f"], latex=True
    )
)
predictions["p_i"] = (
    shoreline.calculate_probability_of_atmosphere_from_posterior_samples(
        log_v=targets["log_v"], log_L=targets["log_L"], log_f=targets["log_f"], latex=False
    )
)[0]

for k in 'vfL':
    predictions[f'log_{k}'].info.format='.3f'

predictions

In [ ]:
print(f'Atmosphere expectation value = {np.sum(predictions['p_i'])}')

Now let's write that table out to AAS LaTeX.

In [ ]:
predictions.rename_column('planet', 'Planet')
predictions.rename_column('log_v', r"$\log_{10}(v_{\rm esc}/v_{esc,\oplus})$")
predictions.rename_column('log_f', r"$\log_{10}(f/f_\oplus)$")
predictions.rename_column('log_L', r"$\log_{10}(L_\star/L_\odot)$")
predictions.rename_column('probability_of_atmosphere', 'Probability of Atmosphere')


predictions.write('latex-posteriors/rocky-worlds-predictions.tex', format='aastex', overwrite=True)

### Let's double check the input parameters

Let's confirm we don't get wildly different answers if we use the current `exoatlas` planet properties instead of the (likely more up-to-date) ones on the STScI website.

In [ ]:
planet_names = list(targets["planet"])
e_rocky_worlds = TransitingExoplanets()[planet_names]

# use "kludge" to estimate escape velocity for planets without published masses
exoatlas_prob = e_rocky_worlds.probability_of_atmosphere(shoreline=shoreline, kludge=True)
stsci_prob = shoreline.calculate_probability_of_atmosphere_from_posterior_samples(
    log_v=targets["log_v"], log_L=targets["log_L"], log_f=targets["log_f"]
)[0]

In [ ]:

plt.scatter(exoatlas_prob, stsci_prob)
for x, y, p in zip(exoatlas_prob, stsci_prob, targets['planet']):
    plt.text(x, y, f'  {p}', ha='left', va='center')
plt.plot([0,1], [0,1])
plt.axis('scaled')
plt.xlabel('Probability of Atmosphere\n(using exoatlas planet properties)')
plt.ylabel('Probability of Atmosphere\n(using STScI planet properties)');

Great, these look close to each other, except for GJ3929b. Diving into the details a little bit, exoatlas appears to be using an old mass for the planet that's too high, thus predicting a higher atmosphere probability than when using the STScI planet mass. Just as an extra check, we'll look below to make sure the inputs generally agree between the two datasets. Indeed, everything seems to line up, except for GJ3929b!

In [ ]:
fi, ax = plt.subplots(1, 3, figsize=(8, 3), constrained_layout=True) 
plt.sca(ax[0]) 
plt.plot([-5, 5],[-5, 5])
plt.scatter(e_rocky_worlds.log_relative_escape_velocity(), targets['log_v'])
plt.text(e_rocky_worlds['GJ3929b'].log_relative_escape_velocity(), targets[0]['log_v'], '  GJ3929b', ha='left', va='center')
plt.axis('scaled')
plt.xlim(0, 0.3)
plt.ylim(0, 0.3)


plt.sca(ax[1]) 
plt.plot([-5, 5],[-5, 5])
plt.scatter(e_rocky_worlds.log_relative_instellation(), targets['log_f'])
plt.axis('scaled')
plt.xlim(-1, 2)
plt.ylim(-1, 2)


plt.sca(ax[2]) 
plt.plot([-5, 5],[-5, 5])
plt.scatter(e_rocky_worlds.log_relative_stellar_luminosity(), targets['log_L'])
plt.axis('scaled')
plt.xlim(-3, 1)
plt.ylim(-3, 1)


## Make plot with these targets!

In [ ]:
pops = load_organized_populations(fluxlimit='any', subset='all')['all+any']

# fix that bad GJ3929 mass, using values from Xue et al (2025)
pops['everything']['transit'].update_values(planets=['GJ3929b'], mass=1.13*u.Mearth, mass_uncertainty=0.09*u.Mearth)
h = [
    "GJ-3929 b",
    "LTT-1445 A c",
    "LTT-1445 A b",
    "LHS-1140 b",
    "TOI-198 b",
    "TOI-406 c",
    "TOI-771 b",
    "HD 260655 c",
    "TOI-244 b",
]
highlights = pops["everything"]["transit"][h]
highlights.color = "orangered"
highlights.filled = False
highlights.outlined = True
highlights.s = 256
highlights.annotate_planets = True
highlights.annotate_kw = dict(rotation=0, rotation_mode="anchor", format="    {}")


teq_limits = [194, 1700] * u.K
flux_limits = 4 * con.sigma_sb * teq_limits**4
earth_insolation = (1 * u.Lsun / 4 / np.pi / u.AU**2).to(u.W / u.m**2)
flim = (flux_limits / earth_insolation).value

m = ShorelineStandardMap(posterior=posterior, Llim=[-1.3, -2.5], flim=np.log10(flim), vlim=[-0.3, 0.7], invisible_fraction=0.8, kludge=True)
g = SliceGridGallery(m, N=3, dpi=300, figsize=(8, 3))
for k, v in clean_pops(pops).items():
    v.annotate_planets = False
    v.label=None
g.build(pops)
highlights.label='Rocky Worlds\nJWST Targets'
g.build(highlights)
g.refine(limits=True, probability=True)
g.maps[2].add_colorbar()
g.maps[1].adjust_annotations(color=highlights.color, alpha=0.5)
g.maps[2].add_legend(loc='upper left', fontsize='x-small', bbox_to_anchor=(0, 0.97))
plt.savefig(f"figures/shoreline-highlighting-rocky-worlds.pdf")

In [ ]:
!cp figures/shoreline-highlighting-rocky-worlds.pdf paper-figures